# 04 Hybrid Ranking - Taxonomy Boost Fixed

Notebook này tạo lại file `13_candidate_job_hybrid_ranking` để so sánh 2 mô hình:

- **Without Taxonomy** = `skill_overlap_score`
- **With Taxonomy** = `0.50 * skill_overlap_score + 0.35 * group_similarity_score + 0.15 * dominant_group_score`

Notebook sẽ tự tìm file baseline/matching có đủ các cột skill/taxonomy trong `ProcessPipeline/data_outputs`, sau đó merge với file semantic similarity ở `Embedding/data_outputs/12_candidate_job_semantic_similarity.xlsx`.


In [50]:
from pathlib import Path
import pandas as pd
import numpy as np

# ===== PATH SETUP =====
PROJECT_DIR = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook")

EMBEDDING_DIR = PROJECT_DIR / "Embedding"
EMBEDDING_OUTPUT_DIR = EMBEDDING_DIR / "data_outputs"

PROCESS_DIR = PROJECT_DIR / "ProcessPipeline"
PROCESS_OUTPUT_DIR = PROCESS_DIR / "data_outputs"

SEMANTIC_PATH = EMBEDDING_OUTPUT_DIR / "12_candidate_job_semantic_similarity.xlsx"

OUTPUT_PARQUET = EMBEDDING_OUTPUT_DIR / "13_candidate_job_hybrid_ranking.parquet"
OUTPUT_EXCEL = EMBEDDING_OUTPUT_DIR / "13_candidate_job_hybrid_ranking.xlsx"

print("PROCESS_OUTPUT_DIR exists:", PROCESS_OUTPUT_DIR.exists(), PROCESS_OUTPUT_DIR)
print("EMBEDDING_OUTPUT_DIR exists:", EMBEDDING_OUTPUT_DIR.exists(), EMBEDDING_OUTPUT_DIR)
print("SEMANTIC_PATH exists:", SEMANTIC_PATH.exists(), SEMANTIC_PATH)


PROCESS_OUTPUT_DIR exists: True /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs
EMBEDDING_OUTPUT_DIR exists: True /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs
SEMANTIC_PATH exists: True /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/12_candidate_job_semantic_similarity.xlsx


In [51]:
# ===== READ SEMANTIC SIMILARITY =====
if not SEMANTIC_PATH.exists():
    raise FileNotFoundError(f"Semantic file not found: {SEMANTIC_PATH}")

semantic_df = pd.read_excel(SEMANTIC_PATH)

print("semantic_df:", semantic_df.shape)
print("semantic columns:")
print(semantic_df.columns.tolist())

required_semantic_cols = ["candidate_id", "job_id", "semantic_similarity"]
missing_semantic_cols = [col for col in required_semantic_cols if col not in semantic_df.columns]

if missing_semantic_cols:
    raise ValueError(f"Missing semantic columns: {missing_semantic_cols}")

semantic_df.head()


semantic_df: (1600, 4)
semantic columns:
['candidate_id', 'job_id', 'semantic_similarity', 'semantic_rank']


,candidate_id,job_id,semantic_similarity,semantic_rank
0,C001,J005,0.906228,1
1,C001,J002,0.893925,2
2,C001,J009,0.887343,3
3,C001,J004,0.886809,4
4,C001,J006,0.880208,5


In [52]:
# ===== AUTO FIND BASELINE / MATCHING FILE =====
# Cần file có đủ các cột:
# candidate_id, job_id, skill_overlap_score, group_similarity_score, dominant_group_score, baseline_score/final_score

required_base_cols = {
    "candidate_id",
    "job_id",
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
}

candidate_files = []

# Tìm trong ProcessPipeline trước
if PROCESS_OUTPUT_DIR.exists():
    candidate_files.extend(list(PROCESS_OUTPUT_DIR.rglob("*.xlsx")))
    candidate_files.extend(list(PROCESS_OUTPUT_DIR.rglob("*.parquet")))

# Tìm thêm trong Embedding để dự phòng
if EMBEDDING_OUTPUT_DIR.exists():
    candidate_files.extend(list(EMBEDDING_OUTPUT_DIR.rglob("*.xlsx")))
    candidate_files.extend(list(EMBEDDING_OUTPUT_DIR.rglob("*.parquet")))

# Loại các file output không nên dùng làm baseline
skip_keywords = [
    "semantic_similarity",
    "hybrid_ranking",
    "candidate_level_metrics",
    "metrics_summary",
    "top5",
    "recommendations",
    "model_comparison",
    "embeddings",
    "candidate_texts",
    "job_texts",
]

valid_baseline_files = []

for path in candidate_files:
    name = path.name.lower()
    if any(key in name for key in skip_keywords):
        continue

    try:
        if path.suffix.lower() == ".xlsx":
            tmp = pd.read_excel(path, nrows=5)
        elif path.suffix.lower() == ".parquet":
            tmp = pd.read_parquet(path)
            tmp = tmp.head(5)
        else:
            continue

        cols = set(tmp.columns.tolist())
        has_required = required_base_cols.issubset(cols)
        has_score = ("baseline_score" in cols) or ("final_score" in cols)

        if has_required and has_score:
            valid_baseline_files.append(path)
    except Exception:
        pass

print("Valid baseline/matching files found:")
for i, path in enumerate(valid_baseline_files):
    print(f"{i}. {path}")

if not valid_baseline_files:
    raise FileNotFoundError(
        "Không tìm thấy file baseline/matching có đủ cột skill_overlap_score, "
        "group_similarity_score, dominant_group_score và baseline_score/final_score. "
        "Hãy kiểm tra lại đã chạy ProcessPipeline chưa."
    )

# Ưu tiên file có tên matching/final
preferred = [
    p for p in valid_baseline_files
    if ("matching" in p.name.lower() or "candidate_job" in p.name.lower())
]

BASELINE_PATH = preferred[0] if preferred else valid_baseline_files[0]

print("\nUsing BASELINE_PATH:", BASELINE_PATH)


Valid baseline/matching files found:
0. /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step06_job_and_matching/07_candidate_job_matching.xlsx

Using BASELINE_PATH: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step06_job_and_matching/07_candidate_job_matching.xlsx


In [53]:
# ===== READ BASELINE / MATCHING FILE =====
if BASELINE_PATH.suffix.lower() == ".xlsx":
    baseline_df = pd.read_excel(BASELINE_PATH)
elif BASELINE_PATH.suffix.lower() == ".parquet":
    baseline_df = pd.read_parquet(BASELINE_PATH)
else:
    raise ValueError(f"Unsupported baseline file type: {BASELINE_PATH.suffix}")

print("baseline_df:", baseline_df.shape)
print("baseline columns:")
print(baseline_df.columns.tolist())

# Nếu file baseline dùng final_score thì đổi thành baseline_score
if "baseline_score" not in baseline_df.columns and "final_score" in baseline_df.columns:
    baseline_df = baseline_df.rename(columns={"final_score": "baseline_score"})

required_baseline_cols = [
    "candidate_id",
    "job_id",
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "baseline_score",
]

missing_baseline_cols = [col for col in required_baseline_cols if col not in baseline_df.columns]
if missing_baseline_cols:
    raise ValueError(f"Missing baseline columns: {missing_baseline_cols}")

baseline_df.head()


baseline_df: (400, 14)
baseline columns:
['candidate_id', 'job_id', 'job_title', 'candidate_mapped_skills', 'job_mapped_skills', 'candidate_groups', 'job_groups', 'candidate_dominant_group', 'job_dominant_group', 'skill_overlap_score', 'group_similarity_score', 'dominant_group_score', 'final_score', 'match_explanation']


,candidate_id,job_id,job_title,candidate_mapped_skills,job_mapped_skills,candidate_groups,job_groups,candidate_dominant_group,job_dominant_group,skill_overlap_score,group_similarity_score,dominant_group_score,baseline_score,match_explanation
0,C001,J042,Automation Test Engineer,"['JavaScript', 'CSS']",['JavaScript'],['Software Development'],['Software Development'],Software Development,Software Development,1.0000,1.0,1,1.0000,skill overlap=1.0; group similarity=1.0; same ...
1,C001,J048,Test Automation Developer,"['JavaScript', 'CSS']","['develop automated software tests', 'JavaScri...",['Software Development'],['Software Development'],Software Development,Software Development,0.5000,1.0,1,0.6750,skill overlap=0.5; group similarity=1.0; same ...
2,C001,J005,React Frontend Developer,"['JavaScript', 'CSS']","['TypeScript', 'implement frontend website des...",['Software Development'],['Software Development'],Software Development,Software Development,0.3333,1.0,1,0.5667,skill overlap=0.3333; group similarity=1.0; sa...
3,C001,J006,Svelte Frontend Developer,"['JavaScript', 'CSS']","['TypeScript', 'implement frontend website des...",['Software Development'],['Software Development'],Software Development,Software Development,0.3333,1.0,1,0.5667,skill overlap=0.3333; group similarity=1.0; sa...
4,C001,J007,Web Component Engineer,"['JavaScript', 'CSS']","['TypeScript', 'implement frontend website des...",['Software Development'],['Software Development'],Software Development,Software Development,0.3333,1.0,1,0.5667,skill overlap=0.3333; group similarity=1.0; sa...


In [54]:
# ===== MERGE BASELINE + SEMANTIC =====
# Lấy các cột cần thiết từ baseline
keep_cols = [
    "candidate_id",
    "job_id",
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "baseline_score",
]

# Giữ thêm cột mô tả nếu có
for optional_col in ["job_title", "match_explanation"]:
    if optional_col in baseline_df.columns:
        keep_cols.append(optional_col)

baseline_small = baseline_df[keep_cols].copy()

semantic_small = semantic_df[[
    "candidate_id",
    "job_id",
    "semantic_similarity",
]].copy()

# Giữ semantic_rank nếu có
if "semantic_rank" in semantic_df.columns:
    semantic_small["semantic_rank"] = semantic_df["semantic_rank"]

hybrid_df = baseline_small.merge(
    semantic_small,
    on=["candidate_id", "job_id"],
    how="left",
)

hybrid_df["semantic_similarity"] = hybrid_df["semantic_similarity"].fillna(0)

print("hybrid_df:", hybrid_df.shape)
print("Candidates:", hybrid_df["candidate_id"].nunique())
print("Jobs:", hybrid_df["job_id"].nunique())

hybrid_df.head()


hybrid_df: (400, 10)
Candidates: 20
Jobs: 53


,candidate_id,job_id,skill_overlap_score,group_similarity_score,dominant_group_score,baseline_score,job_title,match_explanation,semantic_similarity,semantic_rank
0,C001,J042,1.0000,1.0,1,1.0000,Automation Test Engineer,skill overlap=1.0; group similarity=1.0; same ...,0.843510,18
1,C001,J048,0.5000,1.0,1,0.6750,Test Automation Developer,skill overlap=0.5; group similarity=1.0; same ...,0.842319,19
2,C001,J005,0.3333,1.0,1,0.5667,React Frontend Developer,skill overlap=0.3333; group similarity=1.0; sa...,0.906228,1
3,C001,J006,0.3333,1.0,1,0.5667,Svelte Frontend Developer,skill overlap=0.3333; group similarity=1.0; sa...,0.880208,5
4,C001,J007,0.3333,1.0,1,0.5667,Web Component Engineer,skill overlap=0.3333; group similarity=1.0; sa...,0.869554,8


In [55]:
# ===== NUMERIC CLEANING =====
score_cols = [
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "baseline_score",
    "semantic_similarity",
]

for col in score_cols:
    if col not in hybrid_df.columns:
        raise ValueError(f"Missing column: {col}")

    hybrid_df[col] = pd.to_numeric(
        hybrid_df[col],
        errors="coerce"
    ).fillna(0)

# semantic similarity nếu cần dùng về sau
hybrid_df["semantic_score_norm"] = ((hybrid_df["semantic_similarity"] + 1) / 2).clip(0, 1)

# baseline_score có thể đã 0-1
hybrid_df["baseline_score_norm"] = hybrid_df["baseline_score"].clip(0, 1)

print("hybrid_df:", hybrid_df.shape)
print("Candidates:", hybrid_df["candidate_id"].nunique())
print("Jobs:", hybrid_df["job_id"].nunique())

display_cols = [
    "candidate_id",
    "job_id",
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "baseline_score_norm",
    "semantic_similarity",
    "semantic_score_norm",
]

if "job_title" in hybrid_df.columns:
    display_cols.insert(2, "job_title")

hybrid_df[display_cols].head()


hybrid_df: (400, 12)
Candidates: 20
Jobs: 53


,candidate_id,job_id,job_title,skill_overlap_score,group_similarity_score,dominant_group_score,baseline_score_norm,semantic_similarity,semantic_score_norm
0,C001,J042,Automation Test Engineer,1.0000,1.0,1,1.0000,0.843510,0.921755
1,C001,J048,Test Automation Developer,0.5000,1.0,1,0.6750,0.842319,0.921160
2,C001,J005,React Frontend Developer,0.3333,1.0,1,0.5667,0.906228,0.953114
3,C001,J006,Svelte Frontend Developer,0.3333,1.0,1,0.5667,0.880208,0.940104
4,C001,J007,Web Component Engineer,0.3333,1.0,1,0.5667,0.869554,0.934777


In [56]:
# ===== SCORE 2 MODELS =====

# ==========================================================
# MODEL 1: WITHOUT TAXONOMY
# Chỉ dùng skill overlap trực tiếp
# ==========================================================
hybrid_df["hybrid_no_taxonomy_score"] = hybrid_df["skill_overlap_score"].clip(0, 1)

# ==========================================================
# MODEL 2: WITH TAXONOMY - TAXONOMY BOOST MODEL
#
# Công thức:
# with_taxonomy_score =
# 0.50 * skill_overlap_score
# + 0.35 * group_similarity_score
# + 0.15 * dominant_group_score
# ==========================================================
hybrid_df["hybrid_taxonomy_score"] = (
    0.50 * hybrid_df["skill_overlap_score"]
    + 0.35 * hybrid_df["group_similarity_score"]
    + 0.15 * hybrid_df["dominant_group_score"]
).clip(0, 1)

# Giữ lại tên cũ để các notebook/cell phía sau không lỗi
hybrid_df["hybrid_score"] = hybrid_df["hybrid_taxonomy_score"]

display_cols = [
    "candidate_id",
    "job_id",
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "hybrid_no_taxonomy_score",
    "hybrid_taxonomy_score",
    "hybrid_score",
]

if "job_title" in hybrid_df.columns:
    display_cols.insert(2, "job_title")

hybrid_df[display_cols].head(10)


,candidate_id,job_id,job_title,skill_overlap_score,group_similarity_score,dominant_group_score,hybrid_no_taxonomy_score,hybrid_taxonomy_score,hybrid_score
0,C001,J042,Automation Test Engineer,1.0000,1.0,1,1.0000,1.00000,1.00000
1,C001,J048,Test Automation Developer,0.5000,1.0,1,0.5000,0.75000,0.75000
2,C001,J005,React Frontend Developer,0.3333,1.0,1,0.3333,0.66665,0.66665
3,C001,J006,Svelte Frontend Developer,0.3333,1.0,1,0.3333,0.66665,0.66665
4,C001,J007,Web Component Engineer,0.3333,1.0,1,0.3333,0.66665,0.66665
5,C001,J001,Angular Frontend Engineer,0.0000,1.0,1,0.0000,0.50000,0.50000
6,C001,J002,Next.js Web Developer,0.0000,1.0,1,0.0000,0.50000,0.50000
7,C001,J003,Frontend UI Engineer,0.0000,1.0,1,0.0000,0.50000,0.50000
8,C001,J004,Vue Frontend Developer,0.0000,1.0,1,0.0000,0.50000,0.50000
9,C001,J008,Frontend Platform Engineer,0.0000,1.0,1,0.0000,0.50000,0.50000


In [57]:
# ===== CREATE RANKS =====

# Xóa rank cũ nếu chạy lại cell nhiều lần
for col in ["rank_taxonomy", "rank_no_taxonomy", "hybrid_rank"]:
    if col in hybrid_df.columns:
        hybrid_df = hybrid_df.drop(columns=[col])

# Rank model có taxonomy
taxonomy_rank_df = hybrid_df.sort_values(
    ["candidate_id", "hybrid_taxonomy_score"],
    ascending=[True, False]
).copy()

taxonomy_rank_df["rank_taxonomy"] = (
    taxonomy_rank_df.groupby("candidate_id").cumcount() + 1
)

# Rank model không taxonomy
no_taxonomy_rank_df = hybrid_df.sort_values(
    ["candidate_id", "hybrid_no_taxonomy_score"],
    ascending=[True, False]
).copy()

no_taxonomy_rank_df["rank_no_taxonomy"] = (
    no_taxonomy_rank_df.groupby("candidate_id").cumcount() + 1
)

# Merge rank về hybrid_df
hybrid_df = hybrid_df.merge(
    taxonomy_rank_df[["candidate_id", "job_id", "rank_taxonomy"]],
    on=["candidate_id", "job_id"],
    how="left"
)

hybrid_df = hybrid_df.merge(
    no_taxonomy_rank_df[["candidate_id", "job_id", "rank_no_taxonomy"]],
    on=["candidate_id", "job_id"],
    how="left"
)

# Giữ tên cũ để các cell/notebook sau không lỗi
hybrid_df["hybrid_rank"] = hybrid_df["rank_taxonomy"]

hybrid_df = hybrid_df.sort_values(
    ["candidate_id", "rank_taxonomy"],
    ascending=[True, True]
).reset_index(drop=True)

display_cols = [
    "candidate_id",
    "job_id",
    "hybrid_no_taxonomy_score",
    "hybrid_taxonomy_score",
    "rank_no_taxonomy",
    "rank_taxonomy",
]

if "job_title" in hybrid_df.columns:
    display_cols.insert(2, "job_title")

hybrid_df[display_cols].head(20)


,candidate_id,job_id,job_title,hybrid_no_taxonomy_score,hybrid_taxonomy_score,rank_no_taxonomy,rank_taxonomy
0,C001,J042,Automation Test Engineer,1.0000,1.00000,1,1
1,C001,J048,Test Automation Developer,0.5000,0.75000,2,2
2,C001,J005,React Frontend Developer,0.3333,0.66665,3,3
3,C001,J006,Svelte Frontend Developer,0.3333,0.66665,4,4
4,C001,J007,Web Component Engineer,0.3333,0.66665,5,5
5,C001,J001,Angular Frontend Engineer,0.0000,0.50000,6,6
6,C001,J002,Next.js Web Developer,0.0000,0.50000,7,7
7,C001,J003,Frontend UI Engineer,0.0000,0.50000,8,8
8,C001,J004,Vue Frontend Developer,0.0000,0.50000,9,9
9,C001,J008,Frontend Platform Engineer,0.0000,0.50000,10,10


In [58]:
# ===== CHECK TAXONOMY IMPACT =====
hybrid_df["score_diff"] = (
    hybrid_df["hybrid_taxonomy_score"] - hybrid_df["hybrid_no_taxonomy_score"]
)

hybrid_df["rank_diff"] = (
    hybrid_df["rank_no_taxonomy"] - hybrid_df["rank_taxonomy"]
)

num_score_changed = (hybrid_df["score_diff"].abs() > 1e-9).sum()
num_rank_changed = (hybrid_df["rank_diff"] != 0).sum()
num_candidates_rank_changed = hybrid_df.loc[
    hybrid_df["rank_diff"] != 0,
    "candidate_id"
].nunique()

print("Total rows:", len(hybrid_df))
print("Candidates:", hybrid_df["candidate_id"].nunique())
print("Jobs:", hybrid_df["job_id"].nunique())
print("Rows score changed:", num_score_changed)
print("Rows rank changed:", num_rank_changed)
print("Candidates with rank changed:", num_candidates_rank_changed)

display_cols = [
    "candidate_id",
    "job_id",
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "hybrid_no_taxonomy_score",
    "hybrid_taxonomy_score",
    "rank_no_taxonomy",
    "rank_taxonomy",
    "rank_diff",
]

if "job_title" in hybrid_df.columns:
    display_cols.insert(2, "job_title")

hybrid_df[
    hybrid_df["rank_diff"] != 0
][display_cols].head(30)


Total rows: 400
Candidates: 20
Jobs: 53
Rows score changed: 200
Rows rank changed: 73
Candidates with rank changed: 6


,candidate_id,job_id,job_title,skill_overlap_score,group_similarity_score,dominant_group_score,hybrid_no_taxonomy_score,hybrid_taxonomy_score,rank_no_taxonomy,rank_taxonomy,rank_diff
21,C002,J001,Angular Frontend Engineer,0.0000,1.0,1,0.0000,0.50000,4,2,2
22,C002,J002,Next.js Web Developer,0.0000,1.0,1,0.0000,0.50000,5,3,2
23,C002,J003,Frontend UI Engineer,0.0000,1.0,1,0.0000,0.50000,6,4,2
24,C002,J004,Vue Frontend Developer,0.0000,1.0,1,0.0000,0.50000,7,5,2
25,C002,J005,React Frontend Developer,0.0000,1.0,1,0.0000,0.50000,8,6,2
26,C002,J006,Svelte Frontend Developer,0.0000,1.0,1,0.0000,0.50000,9,7,2
27,C002,J007,Web Component Engineer,0.0000,1.0,1,0.0000,0.50000,10,8,2
28,C002,J008,Frontend Platform Engineer,0.0000,1.0,1,0.0000,0.50000,11,9,2
29,C002,J009,UI Dashboard Developer,0.0000,1.0,1,0.0000,0.50000,12,10,2
30,C002,J010,Design System Frontend Engineer,0.0000,1.0,1,0.0000,0.50000,13,11,2


In [59]:
# ===== TAXONOMY CASES =====
taxonomy_cases = hybrid_df[
    (hybrid_df["skill_overlap_score"] < 0.5)
    & (
        (hybrid_df["group_similarity_score"] >= 0.5)
        | (hybrid_df["dominant_group_score"] >= 1)
    )
].copy()

taxonomy_cases = taxonomy_cases.sort_values(
    ["candidate_id", "hybrid_taxonomy_score"],
    ascending=[True, False]
)

print("Taxonomy cases:", taxonomy_cases.shape)

display_cols = [
    "candidate_id",
    "job_id",
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
    "hybrid_no_taxonomy_score",
    "hybrid_taxonomy_score",
    "rank_no_taxonomy",
    "rank_taxonomy",
]

if "job_title" in taxonomy_cases.columns:
    display_cols.insert(2, "job_title")

taxonomy_cases[display_cols].head(30)


Taxonomy cases: (138, 20)


,candidate_id,job_id,job_title,skill_overlap_score,group_similarity_score,dominant_group_score,hybrid_no_taxonomy_score,hybrid_taxonomy_score,rank_no_taxonomy,rank_taxonomy
2,C001,J005,React Frontend Developer,0.3333,1.0,1,0.3333,0.66665,3,3
3,C001,J006,Svelte Frontend Developer,0.3333,1.0,1,0.3333,0.66665,4,4
4,C001,J007,Web Component Engineer,0.3333,1.0,1,0.3333,0.66665,5,5
5,C001,J001,Angular Frontend Engineer,0.0000,1.0,1,0.0000,0.50000,6,6
6,C001,J002,Next.js Web Developer,0.0000,1.0,1,0.0000,0.50000,7,7
7,C001,J003,Frontend UI Engineer,0.0000,1.0,1,0.0000,0.50000,8,8
8,C001,J004,Vue Frontend Developer,0.0000,1.0,1,0.0000,0.50000,9,9
9,C001,J008,Frontend Platform Engineer,0.0000,1.0,1,0.0000,0.50000,10,10
10,C001,J009,UI Dashboard Developer,0.0000,1.0,1,0.0000,0.50000,11,11
11,C001,J010,Design System Frontend Engineer,0.0000,1.0,1,0.0000,0.50000,12,12


In [60]:
# ===== EXPORT =====
required_cols = [
    "hybrid_taxonomy_score",
    "hybrid_no_taxonomy_score",
    "rank_taxonomy",
    "rank_no_taxonomy",
]

missing_cols = [col for col in required_cols if col not in hybrid_df.columns]

if missing_cols:
    raise ValueError(f"Missing columns before export: {missing_cols}")

hybrid_df.to_parquet(OUTPUT_PARQUET, index=False)
hybrid_df.to_excel(OUTPUT_EXCEL, index=False)

print("Saved ->", OUTPUT_PARQUET)
print("Saved ->", OUTPUT_EXCEL)
print("Final shape:", hybrid_df.shape)
print("Candidates:", hybrid_df["candidate_id"].nunique())
print("Jobs:", hybrid_df["job_id"].nunique())


Saved -> /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/13_candidate_job_hybrid_ranking.parquet
Saved -> /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/13_candidate_job_hybrid_ranking.xlsx
Final shape: (400, 20)
Candidates: 20
Jobs: 53


In [61]:
# ===== PREVIEW TOP 10 =====
top10_taxonomy = hybrid_df[hybrid_df["rank_taxonomy"] <= 10].copy()
top10_taxonomy = top10_taxonomy.sort_values(
    ["candidate_id", "rank_taxonomy"],
    ascending=[True, True]
)

top10_no_taxonomy = hybrid_df[hybrid_df["rank_no_taxonomy"] <= 10].copy()
top10_no_taxonomy = top10_no_taxonomy.sort_values(
    ["candidate_id", "rank_no_taxonomy"],
    ascending=[True, True]
)

# Giữ biến cũ nếu notebook sau cần
top5_hybrid = hybrid_df[hybrid_df["rank_taxonomy"] <= 5].copy()

display_cols = [
    "candidate_id",
    "rank_taxonomy",
    "job_id",
    "hybrid_taxonomy_score",
    "skill_overlap_score",
    "group_similarity_score",
    "dominant_group_score",
]

if "job_title" in top10_taxonomy.columns:
    display_cols.insert(3, "job_title")

top10_taxonomy[display_cols].head(30)


,candidate_id,rank_taxonomy,job_id,job_title,hybrid_taxonomy_score,skill_overlap_score,group_similarity_score,dominant_group_score
0,C001,1,J042,Automation Test Engineer,1.00000,1.0000,1.0,1
1,C001,2,J048,Test Automation Developer,0.75000,0.5000,1.0,1
2,C001,3,J005,React Frontend Developer,0.66665,0.3333,1.0,1
3,C001,4,J006,Svelte Frontend Developer,0.66665,0.3333,1.0,1
4,C001,5,J007,Web Component Engineer,0.66665,0.3333,1.0,1
5,C001,6,J001,Angular Frontend Engineer,0.50000,0.0000,1.0,1
6,C001,7,J002,Next.js Web Developer,0.50000,0.0000,1.0,1
7,C001,8,J003,Frontend UI Engineer,0.50000,0.0000,1.0,1
8,C001,9,J004,Vue Frontend Developer,0.50000,0.0000,1.0,1
9,C001,10,J008,Frontend Platform Engineer,0.50000,0.0000,1.0,1
